# Task 2.2 — Reproduction of Core Contribution (OSSVR)
**Paper:** One-Sided Support Vector Regression for Multiclass Cost-Sensitive Classification (Tu & Lin, ICML 2010)  
**Student:** Vaishnav Verma (230160)

## What I Am Reproducing
I reproduce the **One-Sided Support Vector Regression (OSSVR)** algorithm — the core contribution of the paper. This involves:
1. Computing regret vectors from a cost matrix
2. Training K one-sided SVR models (one per class) using the one-sided loss
3. Predicting via argmin over regressor outputs

## Evaluation Metric
**Average misclassification cost:** (1/N) × Σ C[yₙ, ŷₙ], the same primary metric used in the paper.

In [17]:
# === Configuration & Random Seeds ===
import numpy as np
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# === Hyperparameters (all defined in one place) ===
TEST_SIZE = 0.3
C_REG = 100.0       # Regularisation parameter — large C allows tighter fit
GAMMA = 0.5         # RBF kernel parameter — sharper kernel for better discrimination
EPSILON = 0.1       # SVR epsilon (used in two-sided ablation; one-sided sets to 0)

# Cost matrix (same as Task 2.1)
COST_MATRIX = np.array([
    [0, 1, 5],
    [2, 0, 3],
    [10, 1, 0],
])

print("Hyperparameters:")
print(f"  RANDOM_SEED = {RANDOM_SEED}")
print(f"  TEST_SIZE   = {TEST_SIZE}")
print(f"  C_REG       = {C_REG}")
print(f"  GAMMA       = {GAMMA}")
print(f"  EPSILON     = {EPSILON}")
print(f"  COST_MATRIX =\n{COST_MATRIX}")

Hyperparameters:
  RANDOM_SEED = 42
  TEST_SIZE   = 0.3
  C_REG       = 100.0
  GAMMA       = 0.5
  EPSILON     = 0.1
  COST_MATRIX =
[[ 0  1  5]
 [ 2  0  3]
 [10  1  0]]


In [18]:
# === Load and Preprocess Dataset ===
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = load_wine()
X = wine.data
y = wine.target
K = len(np.unique(y))  # number of classes

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

print(f"Training: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples, Classes: {K}")

Training: 124 samples, Test: 54 samples, Classes: 3


The code above loads the Wine dataset, standardises features, and splits into train/test sets. This corresponds to the general experimental setup described in Section 4 of the paper.

In [19]:
# === Step 1: Compute Regret Vectors ===
# For each training example (x_n, y_n), the regret vector is:
#   r_n[k] = C[y_n, k] - C[y_n, y_n] = C[y_n, k]  (since C[y,y] = 0)
# This is the core transformation from Eq. (3)-(4) in the paper.

def compute_regret_vectors(y, cost_matrix):
    """Compute regret vectors for all examples.
    
    Args:
        y: array of true class labels, shape (N,)
        cost_matrix: K x K cost matrix, C[true, pred]
    
    Returns:
        regret: array of shape (N, K) where regret[n, k] = C[y[n], k]
    """
    return cost_matrix[y]  # advanced indexing: selects row y[n] for each example

regret_train = compute_regret_vectors(y_train, COST_MATRIX)
regret_test = compute_regret_vectors(y_test, COST_MATRIX)

print(f"Regret vectors shape: {regret_train.shape}")
print(f"\nExample: y_train[0] = {y_train[0]}")
print(f"  Cost row: C[{y_train[0]}, :] = {COST_MATRIX[y_train[0]]}")
print(f"  Regret:   r[0] = {regret_train[0]}")
print(f"  Correct class regret = {regret_train[0, y_train[0]]} (always 0)")

Regret vectors shape: (124, 3)

Example: y_train[0] = 0
  Cost row: C[0, :] = [0 1 5]
  Regret:   r[0] = [0 1 5]
  Correct class regret = 0 (always 0)


This cell computes the regret vector for every training example. For each example with true class y, the regret of predicting class k is simply C[y, k]. The correct class always has regret 0 (since C[y, y] = 0). This corresponds to **Equations (3)–(4) in Section 3** of the paper, where the authors transform labels into regression targets encoding the cost structure.

In [20]:
# === Step 2: Implement One-Sided SVR (OSSVR) ===
# The one-sided loss only penalises UNDER-prediction: l(f, r) = max(0, r - f)
# This is the key contribution from Eq. (5)-(6) in Section 3.
#
# We solve the DUAL QP:
#   max_alpha  sum_i alpha_i * r_i - (1/2) alpha^T K alpha
#   s.t.  0 <= alpha_i <= C
#
# Then f(x) = sum_i alpha_i * K(x_i, x)   (no bias term — RKHS formulation)
#
# NOTE: The paper's formulation does NOT include a separate bias term.
# Including bias in one-sided SVR leads to a degenerate dual (sum alpha_i = 0
# with alpha_i >= 0 forces all alpha = 0). This is a known property of
# one-sided loss functions.

from sklearn.metrics.pairwise import rbf_kernel
import cvxpy as cp

class OneSidedSVR:
    """One-Sided Support Vector Regression for a single class.
    
    Solves the dual QP:
        max_alpha  r^T alpha - (1/2) alpha^T K alpha
        s.t.  0 <= alpha_i <= C
    
    Prediction: f(x) = sum_i alpha_i * K(x_i, x)
    
    This corresponds to Eq. (5)-(6) in Tu & Lin (ICML 2010).
    """
    
    def __init__(self, C=1.0, gamma='scale'):
        self.C = C
        self.gamma = gamma
        self.alpha_ = None
        self.X_train_ = None
        self.gamma_value_ = None
    
    def _compute_gamma(self, X):
        if self.gamma == 'scale':
            return 1.0 / (X.shape[1] * X.var())
        return self.gamma
    
    def fit(self, X, r):
        """Fit one-sided SVR using the dual QP formulation.
        
        Args:
            X: training features, shape (N, d)
            r: regret targets for this class, shape (N,)
        """
        N = X.shape[0]
        self.X_train_ = X.copy()
        self.gamma_value_ = self._compute_gamma(X)
        
        # Compute kernel matrix
        K_mat = rbf_kernel(X, X, gamma=self.gamma_value_)
        # Ensure positive semi-definiteness
        K_mat = (K_mat + K_mat.T) / 2
        K_mat += 1e-8 * np.eye(N)
        
        # Solve dual QP: max r^T alpha - 0.5 alpha^T K alpha, 0 <= alpha <= C
        alpha = cp.Variable(N)
        objective = cp.Maximize(r @ alpha - 0.5 * cp.quad_form(alpha, K_mat))
        constraints = [alpha >= 0, alpha <= self.C]
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.CLARABEL, verbose=False)
        
        self.alpha_ = np.array(alpha.value).flatten()
        return self
    
    def predict(self, X):
        """Predict regret values for new inputs.
        f(x) = sum_i alpha_i * K(x_i, x)
        """
        K_mat = rbf_kernel(X, self.X_train_, gamma=self.gamma_value_)
        return K_mat @ self.alpha_

print("OneSidedSVR class defined (dual QP via CVXPY).")

OneSidedSVR class defined (dual QP via CVXPY).


This cell implements the **One-Sided SVR** regressor — the core algorithmic contribution of the paper. The key difference from standard SVR is the loss function: instead of the symmetric ε-insensitive loss |r − f|_ε that penalises both over- and under-prediction, OSSVR uses `max(0, r − f)` which only penalises under-prediction. This corresponds to **Equations (5)–(6) in Section 3** of the paper. The regressor uses an RBF kernel and is solved via L-BFGS-B optimisation in the primal, which is equivalent to the dual QP formulated in **Section 3.1**.

In [21]:
# === Step 3: Train K OSSVR Models (One per Class) ===
# Following Sec. 3: train one regressor per class on the corresponding
# column of the regret matrix.

ossvr_models = []
for k in range(K):
    print(f"Training OSSVR for class {k}...")
    model = OneSidedSVR(C=C_REG, gamma=GAMMA)
    model.fit(X_train, regret_train[:, k])
    
    # Training predictions for this class
    train_pred_k = model.predict(X_train)
    train_loss_k = np.mean(np.maximum(0, regret_train[:, k] - train_pred_k))
    print(f"  Mean one-sided loss (train): {train_loss_k:.4f}")
    
    ossvr_models.append(model)

print(f"\nTrained {K} OSSVR models.")

Training OSSVR for class 0...
  Mean one-sided loss (train): 0.0000
Training OSSVR for class 1...
  Mean one-sided loss (train): 0.0000
Training OSSVR for class 2...
  Mean one-sided loss (train): 0.0000

Trained 3 OSSVR models.


This cell trains K = 3 independent OSSVR regressors, one for each class. Each regressor fₖ is trained on the k-th column of the regret matrix — i.e., the regret values for predicting class k across all training examples. This corresponds to the **OVA decomposition** described in **Section 3** of the paper.

In [22]:
# === Step 4: Predict via Argmin ===
# Prediction rule from Sec. 3: y_hat = argmin_k f_k(x)
# The class with the smallest predicted regret is chosen.

def ossvr_predict(models, X):
    """Predict class labels using argmin over OSSVR regressor outputs."""
    predictions = np.column_stack([m.predict(X) for m in models])
    return np.argmin(predictions, axis=1)

# Diagnostic: raw regressor outputs for first 5 test examples
raw_preds = np.column_stack([m.predict(X_test) for m in ossvr_models])
print("Raw regressor outputs for first 10 test examples:")
print("  f_0(x)    f_1(x)    f_2(x)    argmin  true")
for i in range(10):
    pred = np.argmin(raw_preds[i])
    print(f"  {raw_preds[i,0]:8.4f}  {raw_preds[i,1]:8.4f}  {raw_preds[i,2]:8.4f}    {pred}       {y_test[i]}")

y_pred_train = ossvr_predict(ossvr_models, X_train)
y_pred_test = ossvr_predict(ossvr_models, X_test)

print(f"\nTest predictions: {y_pred_test[:10]}")
print(f"True labels:      {y_test[:10]}")

Raw regressor outputs for first 10 test examples:
  f_0(x)    f_1(x)    f_2(x)    argmin  true
    0.0299    0.3246    1.6500    0       0
    0.3132    0.0026    0.4624    1       1
    0.0001    0.3187    1.5945    0       0
    0.0010    0.4905    2.4455    0       0
    0.2912    0.6529    3.4972    0       0
    0.0024    0.7598    3.8017    0       0
    4.9867    0.4982    0.0379    2       2
    0.6536    0.0009    0.9822    1       1
    0.0000    0.0006    0.0029    0       1
    0.7425    0.0707    0.0835    1       2

Test predictions: [0 1 0 0 0 0 2 1 0 1]
True labels:      [0 1 0 0 0 0 2 1 1 2]


This cell implements the **prediction rule** from **Section 3**: ŷ = argminₖ fₖ(x). For each test input, all K regressors produce a predicted regret value, and the class with the minimum predicted regret is selected. This is the key inference step that makes OSSVR a cost-sensitive classifier.

In [23]:
# === Step 5: Evaluate Using Average Misclassification Cost ===
# Primary metric from the paper: (1/N) * sum C[y_true, y_pred]

def avg_misclassification_cost(y_true, y_pred, cost_matrix):
    """Compute average misclassification cost."""
    costs = cost_matrix[y_true, y_pred]
    return np.mean(costs)

train_cost = avg_misclassification_cost(y_train, y_pred_train, COST_MATRIX)
test_cost = avg_misclassification_cost(y_test, y_pred_test, COST_MATRIX)

# Also compute standard accuracy for reference
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print("=" * 50)
print("OSSVR Results")
print("=" * 50)
print(f"Train — Avg Cost: {train_cost:.4f}, Accuracy: {train_acc:.4f}")
print(f"Test  — Avg Cost: {test_cost:.4f}, Accuracy: {test_acc:.4f}")
print("=" * 50)

OSSVR Results
Train — Avg Cost: 0.0000, Accuracy: 1.0000
Test  — Avg Cost: 0.0741, Accuracy: 0.9444


This cell evaluates the OSSVR classifier using the paper's primary metric: **average misclassification cost**. This is computed as (1/N) × Σ C[yₙ, ŷₙ] — the mean cost incurred across all predictions, as defined in **Section 2** of the paper. Standard accuracy is also reported for reference, but cost is the metric the paper optimises for.